# Gain Curve Analysis Notes

This notebook estimates the PMT gain as a function of high voltage using the accepted dark-pulse charge peak. The gain quantity is the single-photoelectron charge, `Q_spe`, converted with

`gain = Q_spe / e`.

The total accepted charge is not used as the gain. Total charge depends on dark-pulse rate, trigger rate, afterpulsing, pickup, and any multi-photoelectron contamination. It is kept as a diagnostic because it helps show when the dark baseline/background is changing.

Important caveat: this workflow assumes the dominant accepted charge peak is the single-photoelectron peak from dark counts. If the accepted spectrum is mostly noise, afterpulses, or multi-PE pulses, the reported gain will be biased.


In [ ]:
# Locate the repository when Jupyter starts in Notebooks/.
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'Notebooks': PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))

import SRC.KingCRAB.gain as gain_helpers
from SRC.KingCRAB.context import configure_module
from SRC.KingCRAB.gain import estimate_spe_charge, fit_multi_pe_charge_spectrum, interp_extrap_log_gain, listed_gain_hint, pedestal_multi_pe_model, plot_charge_spectrum_with_pe_gaussians, process_voltage_folder, trapezoid_integral

from SRC.KingCRAB.pmt import (processing_files, normalize_waveform_times, subtract_baseline, lowpass_filter_waveform, filter_waveforms, extract_observables, integrate_pulse_region, waveform_duration, estimate_charge_peak, average_waveform as average_filtered_waveform)

# -----------------------------
# Imports
# -----------------------------
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

import pandas as pd
from IPython.display import display

## Settings and Inputs

The voltage folders are read from `DATA/Dark_Rates/<voltage>`. The threshold parameters match the PMT processing notebook: the baseline RMS is measured before the pulse, the pulse-height cut keeps waveforms above `THRESHOLD_SIGMA * sigma_mean`, and the charge cut removes the smallest accepted charge integrals.


In [ ]:
# -----------------------------
# Gain curve settings
# -----------------------------
# This notebook lives in CODE. The data lives in the repo-level DATA folder,
# so use paths relative to the notebook's parent project folder.
project_folder = Path.cwd().parent if Path.cwd().name == "CODE" else Path.cwd()
data_folder = project_folder / "DATA"
dark_rates_folder = data_folder / "Dark_Rates"

voltage_values = np.array([300, 400, 500, 600, 700, 800, 900])
voltage_folders = [dark_rates_folder / str(V) for V in voltage_values]

R_TERMINATION = 50.0          # Oscilliscope termination, measured in Ohms
T_PRE_SIGNAL = 20e-9          # seconds, pre-pulse region used to measure baseline and noise
PULSE_START_TIME = 20e-9      # seconds, ignore earlier samples when searching for the PMT pulse
THRESHOLD_SIGMA = 5.0         # pulse-height cut, keeps pulses larger than 5 times the noise
APPLY_FILTER = True           # turn the FFT low-pass filter on or off for all waveforms
F_CUTOFF = 1e9                # Hz, cutoff frequency used only if APPLY_FILTER is True
QC_PERCENTILE = 5             # charge cut, removes the smallest charge pulses after the height cut
E_CHARGE = 1.602176634e-19    # Coulombs, charge of one electron used to convert Q_spe into gain
SPE_HIST_BINS = 80            # number of charge-histogram bins used for charge-spectrum fits
N_PE_MAX = 5                    # maximum PE peak included in the multi-PE fit
MIN_FIT_PULSES = 40             # minimum accepted pulses required to attempt the multi-PE fit

# Build the listed-curve gain hint before processing so results do not depend on cell order.
with (data_folder / "R7378A_Characteristics.json").open("r") as stream:
    _listed_json_for_hint = json.load(stream)
_listed_points_for_hint = np.array([
    point["value"][:2]
    for dataset in _listed_json_for_hint.get("datasetColl", []) if dataset.get("name") == "Gain"
    for point in dataset.get("data", []) if len(point.get("value", [])) >= 2
], dtype=float)
_hint_order = np.argsort(_listed_points_for_hint[:, 0])
_hint_voltage = _listed_points_for_hint[_hint_order, 0]
_hint_gain = _listed_points_for_hint[_hint_order, 1] * 1e13
_hint_coeff = np.polyfit(_hint_voltage, np.log10(_hint_gain), 1)

configure_module(gain_helpers, globals())

## Shared Waveform Processing

These utilities parse segmented oscilloscope text files, normalize each segment's time axis, subtract the pre-pulse baseline, optionally low-pass filter the waveform, and calculate simple observables. The pulse height is used for selection; the final gain comes from integrated charge.


In [ ]:
# -----------------------------
# Utilities copied from PMT_PROCESSING
# -----------------------------














configure_module(gain_helpers, globals())

## Pulse-Region Charge

`integrate_pulse_region` finds the most negative point after `PULSE_START_TIME`, then integrates only the negative-going region around that pulse. This avoids integrating a long baseline region, but it is still a pulse-finding method rather than a full pedestal/SPE model.


In [ ]:
# -----------------------------
# Pulse-region integration copied from PMT_PROCESSING
# -----------------------------

## Multi-PE Charge Fit and Per-Voltage Processing

For each voltage, the notebook builds an accepted charge distribution from baseline-subtracted waveforms. Because a light leak can populate 2 PE, 3 PE, etc., the gain estimate should come from the spacing between PE peaks rather than the largest single histogram peak.

The fitted model is a sum of Gaussian PE peaks whose centers are constrained to `q0 + n * Q_spe`. The fitted `Q_spe` is converted to gain with `gain = Q_spe / e`. The amplitudes of the PE peaks are left free, so the fit can handle non-Poisson PE populations from a light leak.


In [ ]:
# -----------------------------
# Estimate PE peak spacing from a pedestal + multi-PE charge spectrum
# -----------------------------
configure_module(gain_helpers, globals())






# -----------------------------
# Process one voltage folder
# -----------------------------

## Run the Gain Extraction

This loop processes every voltage folder independently and stores one result dictionary per voltage. The printed progress is useful for catching missing folders or unexpectedly empty datasets.


In [ ]:
# -----------------------------
# Run over all voltage folders
# -----------------------------
# This list will hold one dictionary of results for each voltage folder
# Each dictionary contains counts, charges, and the final SPE gain for that voltage
gain_results = []

# Loop over the voltage labels and their matching folders together
# process_voltage_folder does the full waveform processing for one voltage at a time
for voltage, folder in zip(voltage_values, voltage_folders):
    print(f"Processing {voltage} V: {folder}")
    result = process_voltage_folder(folder, voltage)
    gain_results.append(result)

print("Done")


# Extract two additional 900 V points with exactly the same SPE-charge workflow.
terminal_gain_specs = [
    {"label": "Terminal 3, Dark 08_04", "folder": Path("/Volumes/Untitled/Dark 08_04_term 3"), "pattern": "C3C1*.txt", "color": "tab:orange", "marker": "^"},
    {"label": "Terminal 4, Dark 08_04", "folder": Path("/Volumes/Untitled/Dark 08_04"), "pattern": "C4C1*.txt", "color": "tab:green", "marker": "s"},
]
terminal_gain_results = []
for spec in terminal_gain_specs:
    print(f"Processing 900 V {spec['label']}: {spec['folder']}")
    result = process_voltage_folder(spec["folder"], 900, file_pattern=spec["pattern"])
    result.update({key: spec[key] for key in ("label", "color", "marker")})
    terminal_gain_results.append(result)

terminal_gain_table = pd.DataFrame([{
    "Dataset": r["label"],
    "Voltage [V]": r["Voltage [V]"],
    "Files": r["Number of files"],
    "Waveforms": r["Number of waveforms"],
    "SPE charge [C]": r["SPE charge [C]"],
    "SPE charge error [C]": r["SPE charge error [C]"],
    "SPE-fit gain (diagnostic)": r["Mean gain"],
    "SPE-fit gain error": r["Gain error"],
    "Charge-based gain estimate": r["Mean accepted charge [C]"] / E_CHARGE,
    "Charge-based gain error": r["Charge error [C]"] / E_CHARGE,
    "Fit successful": r["PE fit success"],
    "Fit status": r["PE fit message"],
} for r in terminal_gain_results])
display(terminal_gain_table)

## Check the Numerical Results

This table separates the actual gain estimate from background-sensitive diagnostics. `SPE gain` is `SPE charge / e`. `Mean accepted charge`, `charge per waveform`, and accepted rate are useful checks, but they are not themselves PMT gain.


In [ ]:
# -----------------------------
# Print gain table
# -----------------------------
print("========== GAIN CURVE RESULTS ==========")
for result in gain_results:
    print(f"{result['Voltage [V]']:4.0f} V")
    print(f"  Files                      : {result['Number of files']}")
    print(f"  Waveforms                  : {result['Number of waveforms']}")
    print(f"  Baseline mean              : {result['Baseline mean [V]']:.4e} V")
    print(f"  Baseline spread            : {result['Baseline std [V]']:.4e} V")
    print(f"  Mean baseline noise sigma  : {result['Mean noise sigma [V]']:.4e} V")
    print(f"  Accepted pulses            : {result['Accepted pulses N_acc']}")
    print(f"  Accepted fraction          : {result['Accepted fraction']:.4e}")
    print(f"  Accepted pulse rate        : {result['Accepted pulse rate [Hz]']:.4e} Hz")
    print(f"  Mean accepted charge       : {result['Mean accepted charge [C]']:.4e} C")
    print(f"  Charge per waveform        : {result['Charge per waveform [C/wf]']:.4e} C/wf")
    print(f"  SPE charge                 : {result['SPE charge [C]']:.4e} C")
    print(f"  SPE charge error           : {result['SPE charge error [C]']:.4e} C")
    print(f"  SPE peak pulses            : {result['SPE peak pulses']}")
    print(f"  PE fit method              : {result['PE fit method']}")
    print(f"  PE fit status              : {result['PE fit message']}")
    print(f"  SPE gain = Q_spe/e         : {result['Mean gain']:.4e}")
    print(f"  Gain error                 : {result['Gain error']:.4e}")

## Prepare Plot Arrays

The result dictionaries are converted to arrays so invalid gain points can be masked before plotting on a log scale.


In [ ]:
# -----------------------------
# Make arrays for plotting
# -----------------------------
V_curve = np.array([r["Voltage [V]"] for r in gain_results])
G_curve = np.array([r["Mean gain"] for r in gain_results])
G_err = np.array([r["Gain error"] for r in gain_results])
Q_spe_curve = np.array([r["SPE charge [C]"] for r in gain_results])
Q_mean_curve = np.array([r["Mean accepted charge [C]"] for r in gain_results])
Q_per_waveform_curve = np.array([r["Charge per waveform [C/wf]"] for r in gain_results])
N_acc_curve = np.array([r["Accepted pulses N_acc"] for r in gain_results])
accepted_fraction_curve = np.array([r["Accepted fraction"] for r in gain_results])
accepted_rate_curve = np.array([r["Accepted pulse rate [Hz]"] for r in gain_results])
baseline_mean_curve = np.array([r["Baseline mean [V]"] for r in gain_results])
baseline_spread_curve = np.array([r["Baseline std [V]"] for r in gain_results])
noise_sigma_curve = np.array([r["Mean noise sigma [V]"] for r in gain_results])

valid_gain = np.isfinite(V_curve) & np.isfinite(G_curve) & (G_curve > 0)

print(f"Voltages in gain curve = {V_curve[valid_gain]}")
print(f"SPE charges used [C]   = {Q_spe_curve[valid_gain]}")
print(f"Gains in gain curve    = {G_curve[valid_gain]}")

fit_success_curve = np.array([r.get("PE fit success", False) for r in gain_results], dtype=bool)
fallback_curve = valid_gain & (~fit_success_curve)
successful_fit_curve = valid_gain & fit_success_curve

## Listed R7378A Reference Curve

The Hamamatsu/reference gain curve is loaded from digitized JSON data and interpolated in log-gain space. Extrapolated portions are based on a fitted log-linear trend and should be treated as a visual guide.


In [ ]:
# -----------------------------
# Load listed R7378A gain curve from JSON
# -----------------------------
# The JSON came from a log-y plot digitization. Its Gain dataset values are stored
# on the digitized axis scale, so LISTED_GAIN_SCALE converts them onto ordinary PMT gain.
listed_gain_json = data_folder / "R7378A_Characteristics.json"
LISTED_GAIN_SCALE = 1e13

with listed_gain_json.open("r") as f:
    listed_gain_data = json.load(f)

listed_gain_points = []
for dataset in listed_gain_data.get("datasetColl", []):
    if dataset.get("name") == "Gain":
        for point in dataset.get("data", []):
            if "value" in point and len(point["value"]) >= 2:
                listed_gain_points.append(point["value"][:2])

listed_gain_points = np.array(listed_gain_points, dtype=float)
order = np.argsort(listed_gain_points[:, 0])
V_listed = listed_gain_points[order, 0]
G_listed_raw = listed_gain_points[order, 1]
G_listed = G_listed_raw * LISTED_GAIN_SCALE

# Interpolate/extrapolate in log(gain), because PMT gain curves are approximately log-linear.
valid_listed = np.isfinite(V_listed) & np.isfinite(G_listed) & (G_listed > 0)
V_listed = V_listed[valid_listed]
G_listed = G_listed[valid_listed]
logG_listed = np.log10(G_listed)

# Fit log10(gain) vs voltage so the listed curve can be extrapolated below/above
# the digitized voltage range. This keeps the existing interpolation but avoids
# flat extrapolation outside the JSON points.
listed_fit_coeff = np.polyfit(V_listed, logG_listed, deg=1)

configure_module(gain_helpers, globals())

V_min_plot = np.nanmin([np.nanmin(V_curve[valid_gain]), np.nanmin(V_listed)])
V_max_plot = np.nanmax([np.nanmax(V_curve[valid_gain]), np.nanmax(V_listed)])
V_listed_interp = np.linspace(V_min_plot, V_max_plot, 500)
G_listed_interp = interp_extrap_log_gain(V_listed_interp, V_listed, logG_listed, listed_fit_coeff)
G_listed_at_measured = interp_extrap_log_gain(V_curve, V_listed, logG_listed, listed_fit_coeff)

# Smooth experimental curve through the measured points. This is interpolation only
# over the measured voltage range, not an extrapolated model.
valid_exp_interp = valid_gain & np.isfinite(G_curve) & (G_curve > 0)
V_exp = V_curve[valid_exp_interp]
G_exp = G_curve[valid_exp_interp]
exp_order = np.argsort(V_exp)
V_exp = V_exp[exp_order]
G_exp = G_exp[exp_order]
V_exp_interp = np.linspace(np.nanmin(V_exp), np.nanmax(V_exp), 400)
G_exp_interp = 10 ** np.interp(V_exp_interp, V_exp, np.log10(G_exp))

print(f"Loaded listed gain curve from {listed_gain_json}")
print(f"Listed gain voltages: {V_listed}")
print(f"Listed gain values after scale factor {LISTED_GAIN_SCALE:.1e}: {G_listed}")
print(f"Listed log-space extrapolation fit: log10(G) = {listed_fit_coeff[0]:.4e} V + {listed_fit_coeff[1]:.4e}")
print(f"Listed gain interpolated/extrapolated at measured voltages: {G_listed_at_measured[valid_gain]}")

## Final Gain Curve

The measured gain points use `Q_spe / e`. The diagnostic cells below the curve show whether the accepted-pulse population or baseline behavior is changing with voltage. If the accepted charge spectrum does not have a clear SPE-like peak, the gain point should be treated as unreliable even if this plot draws it.


The original voltage-series points use the accepted-charge histogram peak, matching the previously displayed curve. Because the terminal-3/4 multi-PE fits have unacceptable reduced χ², their 900 V markers use the more conservative mean selected pulse charge divided by `e`; the legend distinguishes these charge-based estimates from the SPE-fit curve. The plot limits include padding beyond 900 V and beyond the displayed gain range.

In [ ]:
# -----------------------------
# Plot measured gain curve over listed R7378A curve
# -----------------------------
plt.figure(figsize=(9, 5.5))

plt.plot(
    V_listed_interp,
    G_listed_interp,
    color="tab:blue",
    ls="-",
    lw=2.5,
)
plt.plot(
    V_listed,
    G_listed,
    "s",
    color="tab:cyan",
    markeredgecolor="navy",
    ms=5,
    label="Data sheet",
)
plt.plot(
    V_exp_interp,
    G_exp_interp,
    color="tab:red",
    ls="--",
    lw=2.5,
    label="June Data Run Interpolation",
)
plt.errorbar(
    V_curve[valid_gain],
    G_curve[valid_gain],
    yerr=G_err[valid_gain],
    fmt="o",
    color="black",
    ecolor="tab:orange",
    markerfacecolor="gold",
    markeredgecolor="black",
    markersize=6,
    elinewidth=1.5,
    capsize=4,
    label="June Data run points",
)

for result in terminal_gain_results:
    terminal_gain = result["Mean accepted charge [C]"] / E_CHARGE
    terminal_gain_error = result["Charge error [C]"] / E_CHARGE
    if np.isfinite(terminal_gain) and terminal_gain > 0:
        plt.errorbar(
            result["Voltage [V]"], terminal_gain,
            yerr=terminal_gain_error,
            fmt=result["marker"], color=result["color"],
            markeredgecolor="black", markersize=9,
            elinewidth=1.5, capsize=4,
            label=("Photon Leak Correction July" if "Terminal 3" in result["label"] else "Electrical Noise Correction July"),
            zorder=5,
        )

plt.yscale("log")
plt.xlabel("PMT Voltage [V]")
plt.xlim(280, 940)
plt.ylabel("Gain [anode electrons / photoelectron]")
plt.ylim(8e2, 1.2e7)
plt.title("Measured PMT Gain vs Listed R7378A Gain")
plt.legend()
plt.grid(True, alpha=0.3, which="both")
plt.tight_layout()
plt.show()

In [ ]:
# -----------------------------
# Dark-baseline and charge-population diagnostics
# -----------------------------
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].plot(V_curve, baseline_mean_curve * 1e3, "o-", label="Baseline mean")
axes[0, 0].plot(V_curve, baseline_spread_curve * 1e3, "s-", label="Baseline spread")
axes[0, 0].set_xlabel("PMT Voltage [V]")
axes[0, 0].set_ylabel("Baseline [mV]")
axes[0, 0].set_title("Baseline Offset Diagnostics")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(V_curve, noise_sigma_curve * 1e3, "o-", color="tab:purple")
axes[0, 1].set_xlabel("PMT Voltage [V]")
axes[0, 1].set_ylabel("Baseline noise sigma [mV]")
axes[0, 1].set_title("Pre-Pulse Noise")
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(V_curve, accepted_rate_curve, "o-", color="tab:green", label="Accepted rate")
axes[1, 0].set_yscale("log")
axes[1, 0].set_xlabel("PMT Voltage [V]")
axes[1, 0].set_ylabel("Accepted pulse rate [Hz]")
axes[1, 0].set_title("Dark Pulse Rate Diagnostic")
axes[1, 0].grid(True, alpha=0.3, which="both")

# Raw charge is the mean accepted pulse charge. The Gaussian-fit points are shown
# only where the multi-PE fit succeeded; fallback estimates are marked separately.
axes[1, 1].plot(V_curve, Q_mean_curve, "o-", label="raw charge")
axes[1, 1].plot(V_curve[successful_fit_curve], Q_spe_curve[successful_fit_curve], "s--", label="gaussian fit")
axes[1, 1].plot(V_curve[fallback_curve], Q_spe_curve[fallback_curve], "x", ms=8, mew=2, label="fallback estimate")
if "G_listed_at_measured" in globals():
    axes[1, 1].plot(
        V_curve,
        G_listed_at_measured * E_CHARGE,
        "k--",
        lw=1.8,
        label="PMT data sheet",
    )
axes[1, 1].set_yscale("log")
axes[1, 1].set_xlabel("PMT Voltage [V]")
axes[1, 1].set_ylabel("Charge [C]")
axes[1, 1].set_title("Gain curve")
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3, which="both")

plt.tight_layout()
plt.show()


# -----------------------------
# Dedicated 900 V accepted-charge spectrum with PE Gaussian components
# -----------------------------
configure_module(gain_helpers, globals())


result_900 = next((r for r in gain_results if r["Voltage [V]"] == 900), None)
if result_900 is not None:
    plot_charge_spectrum_with_pe_gaussians(result_900, title="900 V accepted charges with PE Gaussian fit")
else:
    print("No 900 V result found in gain_results")


# Accepted charge spectra for all voltages, with total fit overlaid where available.
valid_spectrum_results = [r for r in gain_results if len(r.get("Accepted charges", []))]
if valid_spectrum_results:
    ncols = 2
    nrows = int(np.ceil(len(valid_spectrum_results) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(12, 3.2 * nrows))
    axes = np.ravel(axes)

    for ax, result in zip(axes, valid_spectrum_results):
        charges = result["Accepted charges"]
        charges = charges[np.isfinite(charges) & (charges > 0)]
        if len(charges):
            ax.hist(charges, bins=SPE_HIST_BINS, histtype="step", lw=1.8, color="black")
            ax.axvline(result["SPE charge [C]"], color="tab:red", ls="--", lw=1.5, label="PE spacing")
            if result.get("PE fit success", False) and len(result.get("PE fit q [C]", [])):
                q_fit = result["PE fit q [C]"]
                ax.plot(q_fit, result["PE fit y"], color="tab:orange", lw=2.0, label="total fit")
                params = result.get("PE fit params")
                if params is not None:
                    q_ped, q_spe = params[0], params[1]
                    for n in range(0, N_PE_MAX + 1):
                        ax.axvline(q_ped + n * q_spe, color="tab:orange", ls=":", lw=1, alpha=0.6)
        ax.set_title(f"{result['Voltage [V]']} V accepted charges")
        ax.set_xlabel("Pulse charge [C]")
        ax.set_ylabel("Counts")
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)

    for ax in axes[len(valid_spectrum_results):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()